# Table Detection Pipeline

This notebook runs the detection and OCR workflow from `detection-model` in a cleaner, interactive form. Update the image name in the configuration cell, then run the cells top to bottom.

In [ ]:
from pathlib import Path
import sys

project_dir = Path.cwd()
if not (project_dir / "config.py").exists():
    candidate = project_dir / "detection-model"
    if (candidate / "config.py").exists():
        project_dir = candidate

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

project_dir

## 1. Imports and shared configuration

The next cell imports only the pieces needed to run the pipeline and defines the image/output paths in one place.

In [ ]:
import json
import logging

from config import TESSERACT_CONFIG
from detector import BorderlessTableDetector
from extraction import extract_table

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

image_dir = project_dir / "test-images"
output_dir = project_dir / "extracted-data"
output_dir.mkdir(exist_ok=True)

available_images = sorted(p.name for p in image_dir.iterdir() if p.is_file())
if not available_images:
    raise FileNotFoundError(f"No images found in {image_dir}")

# Change this to a specific image if needed.
image_name = "5ef068b5-113_table_1.jpg"

In [ ]:
# Pick an image once image_name is set.
image_path = image_dir / image_name
output_image = project_dir / "output.png"
detections_json = project_dir / "detections.json"

image_path, output_image

## 2. Run detection

This cell loads the models, runs the structure detector, and renders the annotated image.

In [ ]:
detector = BorderlessTableDetector(image_path, output_image)
detections, figure = detector.process(
    model_type="structure",
    threshold=0.9,
    show_plot=False,
    save_plot=True,
)

## 3. Extract and preview table data

The extraction cell converts the detected structure into structured rows, then shows the first few results.

In [ ]:
table_data = extract_table(detector, detections)

print("Headers:")
print(table_data.headers)
print()
print("First two rows:")
for row in table_data.rows[:2]:
    print(row)

table_data.rows[:2]

## 4. Optional save

Use this cell to write the extracted result to JSON after you are satisfied with the output.

In [ ]:
save_payload = {
    "image file": str(image_path),
    "ocr configuration": TESSERACT_CONFIG,
    "headers": table_data.headers,
    "rows": table_data.rows,
}

save_path = output_dir / f"extracted_{image_path.stem}.json"
save_path.write_text(json.dumps(save_payload, ensure_ascii=False, indent=2), encoding="utf-8")
save_path